# 02 · Evaluation: per-class metrics, same-split baselines, ablations, class weights

Uses the model saved by notebook 01 (or 08) and re-creates **the identical train / validation / test split** (same seed); the first cells check that the saved test accuracy is reproduced.

**Outputs** under `results/publishability_checks/<dataset>/` and `results/figures/<dataset>/`:
`per_class_metrics_saved_model.csv`, `confusion_matrix_saved_model.csv`, `baseline_comparison_exact_split.csv`,
`baseline_main_parameters_fairness_audit.csv`, `ablation_study.csv`, `class_weight_ablation.csv`, and the corresponding figures.

> **Training rows.** Baselines are trained on at most `BASELINE_SAMPLE_LIMIT` rows, while the selected tree is trained on all of train + validation.
> The `*_FULLTRAIN` rows train default decision trees on the same full data, for a like-for-like tree comparison.

In [ ]:
# ---- Setup: find the package root, work inside results/, import the shared code ----
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "framework.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the paper_package folder.")
    ROOT = ROOT.parent
os.environ["PAPER_PACKAGE_ROOT"] = str(ROOT)

# >>> WHICH DATASET?  nf_ton_iot_v2 | cic_ton_iot | nf_unsw_nb15_v3
# run_all.py sets ACTIVE_DATASET; when running by hand, edit the default below
os.environ.setdefault("ACTIVE_DATASET", "nf_unsw_nb15_v3")

sys.path.insert(0, str(ROOT / "src"))
(ROOT / "results").mkdir(exist_ok=True)
os.chdir(ROOT / "results")            # every output (csv / png / pkl / headers) lands under results/

from run_settings import *            # PAPER_MODE: full | quick | smoke

In [ ]:
from eval_helpers import *
from framework import _json_default, _normalize_col
print('Dataset:', ACTIVE_DATASET)
print('Baselines available -> xgboost:', HAS_XGB, '| lightgbm:', HAS_LGBM, '| seaborn:', HAS_SEABORN)

## Load the saved model and re-create the same split

In [ ]:
artifact, metadata = load_artifact_and_metadata()
ARTIFACT_PATH = metadata["model_path"]
saved_pipeline = artifact["pipeline"]
label_encoder = artifact["label_encoder"]

X, y, le_check, numeric_cols_check = load_publishability_data()
X_train, X_val, X_test, y_train, y_val, y_test, X_final, y_final = same_split(X, y)
class_names = list(le_check.classes_)

print("Loaded data:", X.shape, "| test:", X_test.shape)
print("Classes:", class_names)
print("Best k:", metadata.get("best_k"))
print("Best params:", metadata.get("best_params"))

y_pred_saved = saved_pipeline.predict(X_test)
row = metric_row("saved_model", y_test, y_pred_saved)
print(row)

# guard: the re-created split must reproduce the numbers saved by notebook 01
saved_acc = (metadata.get("test_metrics") or {}).get("accuracy")
if saved_acc is not None and abs(saved_acc - row["accuracy"]) > 1e-6:
    print(f"WARNING: accuracy {row['accuracy']:.6f} differs from metadata {saved_acc:.6f} -> "
          "data file / seed / split settings changed since training!")
else:
    print("split reproduces the saved test accuracy.")

## Confusion matrix and per-class metrics

In [ ]:
cm=confusion_matrix(y_test,y_pred_saved)
cm_df=pd.DataFrame(cm,index=[f'true_{c}' for c in class_names],columns=[f'pred_{c}' for c in class_names])
cm_df.to_csv(PUBLISHABILITY_OUTPUT_DIR/'confusion_matrix_saved_model.csv')
display(cm_df)
prec,rec,f1,support=precision_recall_fscore_support(y_test,y_pred_saved,labels=np.arange(len(class_names)),zero_division=0)
per_class=pd.DataFrame({'class_index':np.arange(len(class_names)),'class_name':class_names,'precision':prec,'recall':rec,'f1':f1,'support':support})
per_class['is_rare']=per_class['class_index'].isin(artifact.get('rare_classes',[]))
per_class.to_csv(PUBLISHABILITY_OUTPUT_DIR/'per_class_metrics_saved_model.csv',index=False)
display(per_class)
print(classification_report(y_test,y_pred_saved,target_names=class_names,zero_division=0))

### Figure — per-class performance

In [ ]:
# =============================================================================
# FIGURE — per-class precision / recall / F1 (rare-class visibility)
# =============================================================================
# Reads the already-saved per-class metrics CSV for the CURRENTLY ACTIVE
# dataset (no retraining). Run once per ACTIVE_DATASET after the
# Publishability-checks section has produced per_class_metrics_saved_model.csv.

per_class_path = PUBLISHABILITY_OUTPUT_DIR / "per_class_metrics_saved_model.csv"
per_class_fig = pd.read_csv(per_class_path).sort_values("support")

fig, ax = plt.subplots(figsize=(max(8, 0.6 * len(per_class_fig)), 5))
x = np.arange(len(per_class_fig))
width = 0.25

ax.bar(x - width, per_class_fig["precision"], width, label="Precision", color="#4C72B0")
ax.bar(x,          per_class_fig["recall"],    width, label="Recall",    color="#DD8452")
ax.bar(x + width,  per_class_fig["f1"],        width, label="F1",        color="#55A868")

ax.set_xticks(x)
ax.set_xticklabels(
    [f"{n}\n(n={s:,})" for n, s in zip(per_class_fig["class_name"], per_class_fig["support"])],
    rotation=45, ha="right", fontsize=8,
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title(f"Per-class test performance — {ACTIVE_DATASET}\n(classes sorted by test-set support, rarest first)")
ax.legend()
ax.axhline(0.0, color="black", linewidth=0.8)
fig.tight_layout()

fig_path = PLOTS_DIR / "per_class_performance.png"
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

### Detection rate per attack type *(binary profiles only)*
The model is trained on the 0/1 label; test rows are then grouped by the evaluation-only `Attack` column (never used for training).
For attack families the rate is the detection rate (recall); for the benign family it is the false-positive rate.

In [ ]:
if ATTACK_COL:
    atk = pd.read_csv(DEFAULT_DATASET_PATH, usecols=[ATTACK_COL], dtype={ATTACK_COL: "category"})[ATTACK_COL].astype(str)
    assert len(atk) == len(X), "row count mismatch between the evaluation column and the feature matrix"
    at_test = atk.loc[X_test.index].to_numpy(); y_te = np.asarray(y_test); pred_mal = (np.asarray(y_pred_saved) == 1)
    rows = []
    for a_ in sorted(set(at_test)):
        m = at_test == a_; n = int(m.sum()); is_benign = float(y_te[m].mean()) < 0.5
        flagged = int(pred_mal[m].sum())
        rows.append({"attack_type": a_, "kind": "benign" if is_benign else "attack", "n_test": n, "flagged_malicious": flagged, "flagged_benign": n - flagged,
                     "rate": flagged / n if not is_benign else flagged / n, "rate_meaning": "false-positive rate" if is_benign else "detection rate"})
    at_df = pd.DataFrame(rows).sort_values(["kind", "n_test"], ascending=[True, False])
    at_df.to_csv(PUBLISHABILITY_OUTPUT_DIR / "attack_type_detection.csv", index=False); display(at_df.round(4))
    atk_only = at_df[at_df.kind == "attack"].sort_values("n_test")
    fig, ax = plt.subplots(figsize=(7.5, max(3, .45 * len(atk_only) + 1.2)))
    colr = ["#c0392b" if r < .5 else "#e67e22" if r < .9 else "#27ae60" for r in atk_only["rate"]]
    ax.barh(atk_only["attack_type"], atk_only["rate"], color=colr)
    for i, (r_, n_) in enumerate(zip(atk_only["rate"], atk_only["n_test"])): ax.text(min(r_ + .01, .98), i, f"{r_:.2f}  (n={n_:,})", va="center", fontsize=7)
    ax.set_xlim(0, 1.2); ax.set_xticks(np.arange(0, 1.01, .2)); ax.set_xlabel("detection rate (recall) per attack type")
    ax.set_title(f"{ACTIVE_DATASET}: which attacks does the binary detector catch?  (benign false-positive rate: {at_df[at_df.kind == 'benign']['rate'].mean():.4f})")
    fig.savefig(PLOTS_DIR / "attack_type_detection.png", dpi=300, bbox_inches="tight"); plt.show()
else:
    print("multi-class profile -> per-attack-type report not needed (the per-class table above already shows it).")

## Same-split comparison: selected tree vs default trees and baseline models

In [ ]:
# Reproducible and fair comparison setup.
# Use None to train baselines on the full train+validation split.
BASELINE_SAMPLE_LIMIT = BASELINE_SAMPLE_LIMIT_CFG
BEST_K = int(metadata['best_k'])
BEST_PARAMS = clean_dt_params(metadata['best_params'])

if BASELINE_SAMPLE_LIMIT and len(X_final) > BASELINE_SAMPLE_LIMIT:
    X_base, _, y_base, _ = train_test_split(
        X_final, y_final, train_size=BASELINE_SAMPLE_LIMIT,
        stratify=y_final, random_state=SEED
    )
    sampling_note = f'stratified training subset: {BASELINE_SAMPLE_LIMIT:,} of {len(X_final):,}'
else:
    X_base, y_base = X_final, y_final
    sampling_note = f'full train+validation split: {len(X_final):,}'

print('Comparison training data:', sampling_note)
print('Held-out test rows:', len(X_test))
print('Optimized DT best_k:', BEST_K)
print('Optimized DT main parameters:', json.dumps(BEST_PARAMS, indent=2, default=str))

# Start with the already fitted optimized model.  It is not retrained here.
saved_extra = {
    'train_seconds': np.nan,
    'predict_seconds': np.nan,
    'n_selected_features': BEST_K,
    'feature_policy': 'optimized selected k',
    'preprocessing': 'SafeFinite + median imputer + RobustScaler + SelectKBest',
    'main_parameters': json.dumps(main_params_for_model(saved_pipeline.named_steps['clf']), default=str),
    'comparison_note': 'final optimized DT artifact; trained on train+validation by optimizer',
}
saved_extra.update(tree_resource_summary(saved_pipeline))
rows = [metric_row('Optimized_DT_HSDO_saved', y_test, y_pred_saved, saved_extra)]

# Baselines: same split, same preprocessing, explicit main parameters.
baseline_defs = [
    ('Dummy_stratified', build_safe_pipeline(DummyClassifier(strategy='stratified', random_state=SEED), k=BEST_K), 'sanity baseline'),
    ('DT_default_all_features', build_safe_pipeline(DecisionTreeClassifier(random_state=SEED), k=None), 'default DT, all numeric features'),
    ('DT_default_best_k', build_safe_pipeline(DecisionTreeClassifier(random_state=SEED), k=BEST_K), 'default DT, same selected feature count'),
    ('DT_balanced_best_k', build_safe_pipeline(DecisionTreeClassifier(random_state=SEED, class_weight='balanced'), k=BEST_K), 'default DT + balanced class weights'),
    ('DT_optimized_params_refit', build_safe_pipeline(DecisionTreeClassifier(random_state=SEED, **BEST_PARAMS), k=BEST_K), 'optimized DT params refit on comparison training data'),
    ('RandomForest_100_best_k', build_safe_pipeline(RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1, class_weight=None), k=BEST_K), 'standard ensemble baseline'),
    ('ExtraTrees_100_best_k', build_safe_pipeline(ExtraTreesClassifier(n_estimators=100, random_state=SEED, n_jobs=-1), k=BEST_K), 'randomized tree ensemble baseline'),
    ('GaussianNB_best_k', build_safe_pipeline(GaussianNB(), k=BEST_K), 'simple probabilistic baseline'),
    ('LogReg_balanced_best_k', build_safe_pipeline(LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1, random_state=SEED), k=BEST_K), 'linear baseline'),
    ('KNN_5_best_k', build_safe_pipeline(KNeighborsClassifier(n_neighbors=5, n_jobs=-1), k=BEST_K), 'non-parametric baseline'),
    ('HistGradientBoosting_best_k', build_safe_pipeline(HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=SEED), k=BEST_K), 'sklearn boosting baseline'),
]
if HAS_XGB:
    baseline_defs.append((
        'XGBoost_200_best_k',
        build_safe_pipeline(XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.1, eval_metric='mlogloss', tree_method='hist', random_state=SEED), k=BEST_K),
        'optional gradient boosting baseline'
    ))
if HAS_LGBM:
    baseline_defs.append((
        'LightGBM_300_best_k',
        build_safe_pipeline(LGBMClassifier(n_estimators=300, random_state=SEED), k=BEST_K),
        'optional gradient boosting baseline'
    ))

for name, pipe, note in baseline_defs:
    print('training', name)
    try:
        n_feats = BEST_K if ('selector' in pipe.named_steps) else len(numeric_cols_check)
        model = pipe.named_steps['clf']
        meta_extra = {
            'n_selected_features': n_feats,
            'feature_policy': 'best_k' if ('selector' in pipe.named_steps) else 'all numeric features',
            'preprocessing': 'SafeFinite + median imputer + RobustScaler' + (' + SelectKBest' if 'selector' in pipe.named_steps else ''),
            'main_parameters': json.dumps(main_params_for_model(model), default=str),
            'comparison_note': note,
        }
        row, pred, fitted = evaluate_model(name, pipe, X_base, y_base, X_test, y_test, meta_extra)
        rows.append(row)
    except Exception as exc:
        print(f'  SKIPPED {name}: {exc}')
        rows.append({
            'model': name, 'accuracy': np.nan, 'macro_f1': np.nan, 'weighted_f1': np.nan,
            'mcc': np.nan, 'kappa': np.nan, 'train_seconds': np.nan, 'predict_seconds': np.nan,
            'node_count': np.nan, 'leaf_count': np.nan, 'depth': np.nan, 'size_proxy_kb': np.nan,
            'n_selected_features': BEST_K if 'best_k' in name else np.nan,
            'feature_policy': 'best_k' if 'best_k' in name else 'varies',
            'preprocessing': 'same safe pipeline when fit succeeds',
            'main_parameters': '{}',
            'comparison_note': f'blocked/skipped: {exc}',
        })

# ---- like-for-like decision-tree rows trained on the same (full) train+val data as the selected tree ----
# (the loop above trains the baselines on at most BASELINE_SAMPLE_LIMIT rows, the optimized tree on all of
#  train+val; these rows remove that difference for the DT-vs-DT comparison)
if BASELINE_SAMPLE_LIMIT and len(X_final) > BASELINE_SAMPLE_LIMIT:
    for name, pipe, note in [
        ('DT_default_all_features_FULLTRAIN', build_safe_pipeline(DecisionTreeClassifier(random_state=SEED), k=None), 'default DT, all features, full train+val'),
        ('DT_default_best_k_FULLTRAIN', build_safe_pipeline(DecisionTreeClassifier(random_state=SEED), k=BEST_K), 'default DT, same K, full train+val'),
        ('DT_optimized_params_refit_FULLTRAIN', build_safe_pipeline(DecisionTreeClassifier(random_state=SEED, **BEST_PARAMS), k=BEST_K), 'optimized params refit, full train+val (should equal the saved model)'),
    ]:
        print('training', name)
        mdl = pipe.named_steps['clf']
        extra = {'n_selected_features': BEST_K if 'selector' in pipe.named_steps else len(numeric_cols_check),
                 'feature_policy': 'best_k' if 'selector' in pipe.named_steps else 'all numeric features',
                 'preprocessing': 'SafeFinite + median imputer + RobustScaler' + (' + SelectKBest' if 'selector' in pipe.named_steps else ''),
                 'main_parameters': json.dumps(main_params_for_model(mdl), default=str), 'comparison_note': note}
        r, _, _ = evaluate_model(name, pipe, X_final, y_final, X_test, y_test, extra)
        rows.append(r)

baseline_df = pd.DataFrame(rows).sort_values('mcc', ascending=False, na_position='last')
baseline_csv = PUBLISHABILITY_OUTPUT_DIR / 'baseline_comparison_exact_split.csv'
baseline_df.to_csv(baseline_csv, index=False)
print('Saved baseline comparison:', baseline_csv)
display(baseline_df)

# Parameter audit table (which settings each baseline used).
parameter_audit = baseline_df[[
    'model', 'n_selected_features', 'feature_policy', 'preprocessing',
    'main_parameters', 'comparison_note'
]].copy()
parameter_audit_csv = PUBLISHABILITY_OUTPUT_DIR / 'baseline_main_parameters_fairness_audit.csv'
parameter_audit.to_csv(parameter_audit_csv, index=False)
print('Saved fairness parameter audit:', parameter_audit_csv)
display(parameter_audit)

# Publication-quality figures.
plot_paths = {}
plot_paths['mcc_bar'] = str(save_bar_plot(baseline_df, 'mcc', 'Held-out test MCC: optimized DT vs baselines', 'baseline_mcc_bar.png'))
plot_paths['macro_f1_bar'] = str(save_bar_plot(baseline_df, 'macro_f1', 'Held-out test macro-F1: optimized DT vs baselines', 'baseline_macro_f1_bar.png'))
plot_paths['predict_time_bar'] = str(save_bar_plot(baseline_df.fillna({'predict_seconds': 0}), 'predict_seconds', 'Prediction time on held-out test split', 'baseline_predict_time_bar.png'))
plot_paths['mcc_vs_size'] = str(save_scatter_plot(baseline_df, 'size_proxy_kb', 'mcc', 'MCC vs decision-tree size proxy (DT models only)', 'baseline_mcc_vs_size_proxy.png'))

# Confusion matrix heatmap for optimized model.
fig, ax = plt.subplots(figsize=(max(7, 0.55 * len(class_names)), max(5, 0.45 * len(class_names))))
if HAS_SEABORN:
    sns.heatmap(cm_df, cmap='Blues', ax=ax, cbar=True)
else:
    ax.imshow(cm_df.values, cmap='Blues')
    ax.set_xticks(range(len(cm_df.columns)), cm_df.columns, rotation=90)
    ax.set_yticks(range(len(cm_df.index)), cm_df.index)
ax.set_title('Optimized DT confusion matrix on held-out test set')
ax.set_xlabel('Predicted class')
ax.set_ylabel('True class')
fig.tight_layout()
cm_plot = PLOTS_DIR / 'optimized_dt_confusion_matrix_heatmap.png'
fig.savefig(cm_plot, dpi=300, bbox_inches='tight')
plt.show()
plot_paths['confusion_matrix_heatmap'] = str(cm_plot)

# Optional tree visualization: top levels only, because a full tree can be unreadable.
try:
    fig, ax = plt.subplots(figsize=(18, 8))
    plot_tree(
        saved_pipeline.named_steps['clf'],
        max_depth=3,
        filled=True,
        rounded=True,
        class_names=[str(c) for c in class_names],
        ax=ax,
        fontsize=7,
    )
    ax.set_title('Optimized DT structure, top 3 levels')
    fig.tight_layout()
    tree_plot = PLOTS_DIR / 'optimized_dt_top3_tree.png'
    fig.savefig(tree_plot, dpi=300, bbox_inches='tight')
    plt.show()
    plot_paths['optimized_tree_top3'] = str(tree_plot)
except Exception as exc:
    print('Tree plot skipped:', exc)

with open(PUBLISHABILITY_OUTPUT_DIR / 'plot_paths.json', 'w', encoding='utf-8') as f:
    json.dump(plot_paths, f, indent=2)
print('Saved plot manifest:', PUBLISHABILITY_OUTPUT_DIR / 'plot_paths.json')
print(json.dumps(plot_paths, indent=2))

## Ablation: number of selected features and pruning

In [ ]:
ABLATION_SAMPLE_LIMIT=ABLATION_SAMPLE_LIMIT_CFG  # set None for full train+val
if ABLATION_SAMPLE_LIMIT and len(X_final)>ABLATION_SAMPLE_LIMIT:
    X_ab,_,y_ab,_=train_test_split(X_final,y_final,train_size=ABLATION_SAMPLE_LIMIT,stratify=y_final,random_state=SEED)
else:
    X_ab,y_ab=X_final,y_final
best_params=clean_dt_params(metadata['best_params'])
ablation_defs=[]
for k in ABLATION_KS_CFG:
    ablation_defs.append((f'final_params_k{k}',build_safe_pipeline(DecisionTreeClassifier(random_state=SEED,**best_params),k=k)))
ablation_defs.append(('final_params_no_feature_selection',build_safe_pipeline(DecisionTreeClassifier(random_state=SEED,**best_params),k=None)))
less_pruned=dict(best_params); less_pruned['ccp_alpha']=0.0; less_pruned['max_depth']=None
ablation_defs.append((f"less_pruned_k{metadata['best_k']}",build_safe_pipeline(DecisionTreeClassifier(random_state=SEED,**less_pruned),k=metadata['best_k'])))
rows=[]
for name,pipe in ablation_defs:
    print('ablation',name)
    row,pred,fitted=evaluate_model(name,pipe,X_ab,y_ab,X_test,y_test)
    tree=fitted.named_steps['clf']; row.update({'node_count':tree.tree_.node_count,'leaf_count':tree.get_n_leaves(),'depth':tree.get_depth(),'size_proxy_kb':tree.tree_.node_count*EMBEDDED_BYTES_PER_NODE_EST/1024})
    rows.append(row)
ablation_df=pd.DataFrame(rows).sort_values('mcc',ascending=False)
ablation_df.to_csv(PUBLISHABILITY_OUTPUT_DIR/'ablation_study.csv',index=False)
display(ablation_df)
print('True no-rare-class-fitness requires rerunning the optimizer with rare-class weights/bonus set to zero.')

### Figure — feature-selection trade-off

In [ ]:
# =============================================================================
# FIGURE — feature-selection / model-size tradeoff (ablation across k)
# =============================================================================
# Reads the already-saved ablation_study.csv for the CURRENTLY ACTIVE dataset
# (no retraining). Run once per ACTIVE_DATASET after the ablation-study cell
# has produced ablation_study.csv.

ablation_path = PUBLISHABILITY_OUTPUT_DIR / "ablation_study.csv"
ablation_fig_df = pd.read_csv(ablation_path)

# Keep only the "final_params_kN" rows for the k-sweep line (drop the
# no-feature-selection / less-pruned variants, which don't have a clean k).
k_rows = ablation_fig_df[ablation_fig_df["model"].str.match(r"final_params_k\d+$")].copy()
k_rows["k"] = k_rows["model"].str.extract(r"k(\d+)$").astype(int)
k_rows = k_rows.sort_values("k")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].plot(k_rows["k"], k_rows["mcc"], marker="o", label="MCC", color="#4C72B0")
axes[0].plot(k_rows["k"], k_rows["macro_f1"], marker="s", label="Macro-F1", color="#55A868")
axes[0].set_xlabel("Number of selected features (k)")
axes[0].set_ylabel("Score")
axes[0].set_title(f"Accuracy vs. feature count — {ACTIVE_DATASET}")
axes[0].legend()
axes[0].set_ylim(0, 1)

axes[1].plot(k_rows["k"], k_rows["size_proxy_kb"], marker="o", color="#C44E52")
axes[1].set_xlabel("Number of selected features (k)")
axes[1].set_ylabel("Embedded size proxy (KB)")
axes[1].set_title(f"Model size vs. feature count — {ACTIVE_DATASET}")

fig.suptitle("Feature-selection tradeoff (accuracy vs. embedded footprint)", fontsize=12)
fig.tight_layout()

fig_path = PLOTS_DIR / "ablation_feature_tradeoff.png"
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

## Class-weight ablation (imbalance handling without resampling)

In [ ]:
# ================================================================
# CLASS-WEIGHT ABLATION -- LEAKAGE-SAFE  (no SMOTE, no undersampling)
# class_weight=None vs "balanced"; the variant is selected by VALIDATION MCC only.
# ================================================================
from sklearn.base import clone

def run_class_weight_variant(weight):
    pipe = clone(saved_pipeline)
    pipe.named_steps["clf"].set_params(class_weight=weight)
    pipe.fit(X_train, y_train)                       # training split only
    val_pred, test_pred = pipe.predict(X_val), pipe.predict(X_test)
    tree = pipe.named_steps["clf"]
    return {
        "class_weight": str(weight),
        "val_accuracy": accuracy_score(y_val, val_pred),
        "val_macro_f1": f1_score(y_val, val_pred, average="macro", zero_division=0),
        "val_mcc": matthews_corrcoef(y_val, val_pred),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "test_macro_f1": f1_score(y_test, test_pred, average="macro", zero_division=0),
        "test_weighted_f1": f1_score(y_test, test_pred, average="weighted", zero_division=0),
        "test_mcc": matthews_corrcoef(y_test, test_pred),
        "test_kappa": cohen_kappa_score(y_test, test_pred),
        "node_count": int(tree.tree_.node_count), "depth": int(tree.get_depth()),
        "pipeline": pipe, "test_pred": test_pred,
    }

cw_models, rows_cw = {}, []
for weight in CLASS_WEIGHT_OPTIONS:
    r = run_class_weight_variant(weight)
    cw_models[str(weight)] = r
    rows_cw.append({k: v for k, v in r.items() if k not in ("pipeline", "test_pred")})
cw_df = pd.DataFrame(rows_cw)
cw_df.to_csv(PUBLISHABILITY_OUTPUT_DIR / "class_weight_ablation.csv", index=False)
display(cw_df)

best_key = cw_df.loc[cw_df["val_mcc"].idxmax(), "class_weight"]
print("Selected by validation MCC:", best_key)

# per-class recall of both variants (what the imbalance discussion in the paper needs)
rec_rows = []
for key, r in cw_models.items():
    p_, rc_, f_, s_ = precision_recall_fscore_support(y_test, r["test_pred"], labels=np.arange(len(class_names)), zero_division=0)
    for i, c in enumerate(class_names):
        rec_rows.append({"class_weight": key, "class_name": c, "recall": rc_[i], "f1": f_[i], "support": int(s_[i])})
cw_pc = pd.DataFrame(rec_rows)
cw_pc.to_csv(PUBLISHABILITY_OUTPUT_DIR / "class_weight_per_class.csv", index=False)

piv = cw_pc.pivot(index="class_name", columns="class_weight", values="recall").reindex(class_names)
fig, ax = plt.subplots(figsize=(max(7, .8 * len(class_names)), 3.8))
piv.plot.bar(ax=ax, width=.8)
ax.set_ylabel("test recall"); ax.set_xlabel(""); ax.set_ylim(0, 1.05)
ax.set_title(f"{ACTIVE_DATASET}: per-class recall, class_weight=None vs 'balanced'")
plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
fig.savefig(PLOTS_DIR / "class_weight_per_class_recall.png", dpi=300, bbox_inches="tight"); plt.show()

## ESP32 checklist and evaluation summary

In [ ]:
esp32_measurements = pd.DataFrame([
    {'measurement': 'compiled_flash_bytes', 'value': 'fill in: Arduino/ESP-IDF build output'},
    {'measurement': 'static_ram_bytes', 'value': 'fill in: build map / size output'},
    {'measurement': 'heap_free_before_after', 'value': 'fill in: ESP.getFreeHeap() before/after'},
    {'measurement': 'latency_us_per_sample', 'value': 'fill in: micros() around predict over many samples'},
    {'measurement': 'accuracy_after_conversion', 'value': 'fill in: compare C predictions against sklearn on exported test subset'},
])
esp32_measurements.to_csv(ESP32_DIR / 'esp32_measurement_checklist.csv', index=False)
display(esp32_measurements)
print('Saved:', ESP32_DIR / 'esp32_measurement_checklist.csv')

In [ ]:
summary = {
    'saved_model_metrics': metric_row('saved_model', y_test, y_pred_saved, tree_resource_summary(saved_pipeline)),
    'fairness_statement': (
        'Baseline models are evaluated on the same stratified held-out test split. '
        'Unless explicitly named all_features, baselines use the optimized model best_k feature count '
        'inside the same leakage-safe preprocessing pipeline.'
    ),
    'outputs': {
        'confusion_matrix': str(PUBLISHABILITY_OUTPUT_DIR / 'confusion_matrix_saved_model.csv'),
        'per_class_metrics': str(PUBLISHABILITY_OUTPUT_DIR / 'per_class_metrics_saved_model.csv'),
        'baseline_comparison': str(PUBLISHABILITY_OUTPUT_DIR / 'baseline_comparison_exact_split.csv'),
        'baseline_parameter_audit': str(PUBLISHABILITY_OUTPUT_DIR / 'baseline_main_parameters_fairness_audit.csv'),
        'ablation_study': str(PUBLISHABILITY_OUTPUT_DIR / 'ablation_study.csv'),
        'plot_manifest': str(PUBLISHABILITY_OUTPUT_DIR / 'plot_paths.json'),
        'esp32_header': str(ESP32_DIR / 'esp32_dt_model.h'),
        'esp32_measurement_checklist': str(ESP32_DIR / 'esp32_measurement_checklist.csv'),
    },
    'size_note': (
        'Size values here are host-side estimates; flash, RAM and latency are measured on the board '
        '(notebooks 03 and 05).'
    ),
}
with open(PUBLISHABILITY_OUTPUT_DIR / 'evaluation_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, default=_json_default)
print(json.dumps(summary, indent=2, default=_json_default))

In [ ]:
# Article-ready summary. This reports existing artifacts; it does not invent results.
from pathlib import Path
import json
import pandas as pd

ARTICLE_OUTPUT_DIR = Path('article_results') / ACTIVE_DATASET
ARTICLE_OUTPUT_DIR.mkdir(exist_ok=True)

article_summary = {
    'status': 'generated_from_notebook_artifacts',
    'dataset_profile': ACTIVE_DATASET,
    'dataset_path': str(DEFAULT_DATASET_PATH),
    'target_column': TARGET_COL,
    'random_seed': SEED,
    'split_policy': 'stratified train/validation/test; test used once at final evaluation',
    'class_names': [str(x) for x in le_check.classes_],
    'n_total': int(len(y)),
    'n_train': int(len(y_train)),
    'n_validation': int(len(y_val)),
    'n_test': int(len(y_test)),
    'validation_selected_k': int(metadata.get('best_k', -1)),
    'validation_selected_params': metadata.get('best_params', {}),
    'validation_metrics': metadata.get('validation_metrics', {}),
    'held_out_test_metrics': metadata.get('test_metrics', {}),
    'hardware_validation_status': 'not_run_until_physical_ESP32_measurement',
    'provenance_note': 'All reported numerical values must be taken from the saved CSV/JSON artifacts in this run.'
}

with open(ARTICLE_OUTPUT_DIR / 'article_results_summary.json', 'w', encoding='utf-8') as f:
    json.dump(article_summary, f, indent=2, default=str)

# Record the paths of the summary artifacts that exist.
artifact_manifest = {}
for name in [
    'baseline_comparison_exact_split.csv',
    'baseline_main_parameters_fairness_audit.csv',
    'ablation_study.csv',
    'confusion_matrix_saved_model.csv',
    'per_class_metrics_saved_model.csv',
    'esp32_dt_model.h',
    'esp32_measurement_checklist.csv',
    'evaluation_summary.json',
]:
    q = PUBLISHABILITY_OUTPUT_DIR / name
    artifact_manifest[name] = {'path': str(q), 'exists': bool(q.exists())}
with open(ARTICLE_OUTPUT_DIR / 'article_artifact_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(artifact_manifest, f, indent=2)

print(json.dumps(article_summary, indent=2, default=str))
print('Saved:', ARTICLE_OUTPUT_DIR / 'article_results_summary.json')
print('Saved:', ARTICLE_OUTPUT_DIR / 'article_artifact_manifest.json')

In [ ]:
# Create a hardware measurement template. Fill values only after running the
# generated sketch on the physical ESP32 board.
hardware_template = pd.DataFrame([
    {'measurement': 'board_model', 'value': 'fill in'},
    {'measurement': 'framework_and_version', 'value': 'Arduino/ESP-IDF version: fill in'},
    {'measurement': 'cpu_frequency_mhz', 'value': 'fill in'},
    {'measurement': 'compiler_and_optimization', 'value': 'fill in'},
    {'measurement': 'compiled_flash_bytes', 'value': 'fill in from build output'},
    {'measurement': 'compiled_static_ram_bytes', 'value': 'fill in from build output'},
    {'measurement': 'free_heap_before_bytes', 'value': 'fill in from ESP.getFreeHeap()'},
    {'measurement': 'free_heap_after_bytes', 'value': 'fill in from ESP.getFreeHeap()'},
    {'measurement': 'warmup_iterations', 'value': 'fill in'},
    {'measurement': 'timed_iterations', 'value': 'fill in'},
    {'measurement': 'latency_median_us', 'value': 'fill in'},
    {'measurement': 'latency_p95_us', 'value': 'fill in'},
    {'measurement': 'latency_max_us', 'value': 'fill in'},
    {'measurement': 'python_esp32_prediction_agreement', 'value': 'fill in'},
    {'measurement': 'hardware_status', 'value': 'not_run'},
])
hardware_template.to_csv(ARTICLE_OUTPUT_DIR / 'esp32_hardware_measurement_template.csv', index=False)
display(hardware_template)
print('Saved:', ARTICLE_OUTPUT_DIR / 'esp32_hardware_measurement_template.csv')

**Next:** `03_esp32_export.ipynb` for the same dataset (C header, test vectors and benchmark sketch).